# Problem Statement

**Predict product demand using pricing, promotions, and seasonal factors to help optimize inventory and revenue planning.**




## Business Objective

The goal of this project is to predict product demand to help optimize inventory levels, reduce stockouts, and improve revenue forecasting.

Accurate demand prediction allows:
- Better inventory planning
- Reduced holding costs
- Improved pricing strategy

In [1]:
import pandas as pd
train = pd.read_csv('train.csv')
train.head()

,date,store,item,sales
0,2013-01-01,1,1.0,13.0
1,2013-01-02,1,1.0,11.0
2,2013-01-03,1,1.0,14.0
3,2013-01-04,1,1.0,13.0
4,2013-01-05,1,1.0,10.0


In [2]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500739 entries, 0 to 500738
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   date    500739 non-null  object 
 1   store   500739 non-null  int64  
 2   item    500738 non-null  float64
 3   sales   500738 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 15.3+ MB


In [3]:
train.describe()

,store,item,sales
count,500739.000000,500738.000000,500738.000000
mean,5.455826,14.215786,54.099347
std,2.875866,7.918722,29.953233
min,1.000000,1.000000,0.000000
25%,3.000000,7.000000,30.000000
50%,5.000000,14.000000,49.000000
75%,8.000000,21.000000,72.000000
max,10.000000,28.000000,231.000000


In [4]:
test = pd.read_csv('test.csv')
test.head()

,id,date,store,item
0,0,2018-01-01,1,1
1,1,2018-01-02,1,1
2,2,2018-01-03,1,1
3,3,2018-01-04,1,1
4,4,2018-01-05,1,1


In [5]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      45000 non-null  int64 
 1   date    45000 non-null  object
 2   store   45000 non-null  int64 
 3   item    45000 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 1.4+ MB


In [6]:
test.describe()

,id,store,item
count,45000.000000,45000.000000,45000.00000
mean,22499.500000,5.500000,25.50000
std,12990.525394,2.872313,14.43103
min,0.000000,1.000000,1.00000
25%,11249.750000,3.000000,13.00000
50%,22499.500000,5.500000,25.50000
75%,33749.250000,8.000000,38.00000
max,44999.000000,10.000000,50.00000


In [7]:
train.shape

(500739, 4)

In [8]:
test.shape

(45000, 4)

After looking at the data, it looks like we have 3 input variable -> [date, store, item] and our target Variable is -> [Sales]. Lets do some feature engineering on them.

In [9]:
train['date'] = pd.to_datetime(train['date'])
train = train.sort_values(['store', 'item', 'date'])
test['date'] = pd.to_datetime(test['date'])

In [10]:
lags = [1, 7, 30]
for lag in lags:
    # lag_sales_1: What you sold yesterday, lag_sales_7: What you sold exactly one week ago (the same day of the week) single snapshot, lag_sales_30: What you sold one month ago.
    train[f'lag_sales_{lag}'] = train.groupby(['store', 'item'])['sales'].shift(lag)

# 7-day and 30-day windows are most common for retail
#For every single day in your data, you have a number representing "How well did we do over the last 7 days?" Adding up sales from the last 7 days and dividing by 7. If you sold 10, 12, 10, 11, 9, 10, and 12, your mean is 10.5.
train['rolling_mean_7'] = train.groupby(['store', 'item'])['sales'] \
                         .transform(lambda x: x.rolling(window=7, min_periods=1).mean())
#For every single day in your data, you have a number representing "How well did we do over the last 30 days?"
train['rolling_mean_30'] = train.groupby(['store', 'item'])['sales'] \
                          .transform(lambda x: x.rolling(window=30, min_periods=1).mean())
# Calculate rolling standard deviation (How spread the data is for a week, if it is smooth or bumpy, if this is low that means sales are steady and predictable, if high means maybe bursty or some promotions or holiday.)
train['rolling_std_7'] = train.groupby(['store', 'item'])['sales'] \
                        .transform(lambda x: x.rolling(window=7, min_periods=1).std())

# Flag as promo if sales > (Rolling Mean + 1.5 * Rolling Std)
train['is_promotion'] = (train['sales'] > (train['rolling_mean_7'] + 1.5 * train['rolling_std_7'])).astype(int)
train.dropna(inplace=True)
train

,date,store,item,sales,lag_sales_1,lag_sales_7,lag_sales_30,rolling_mean_7,rolling_mean_30,rolling_std_7,is_promotion
30,2013-01-31,1,1.0,13.0,9.0,8.0,13.0,11.000000,10.500000,2.708013,0
31,2013-02-01,1,1.0,11.0,13.0,14.0,11.0,10.571429,10.500000,2.370453,0
32,2013-02-02,1,1.0,21.0,11.0,12.0,14.0,11.857143,10.733333,4.634241,1
33,2013-02-03,1,1.0,15.0,21.0,12.0,13.0,12.285714,10.800000,4.785892,0
34,2013-02-04,1,1.0,14.0,15.0,11.0,10.0,12.714286,10.933333,4.785892,0
...,...,...,...,...,...,...,...,...,...,...,...
493015,2017-12-27,10,27.0,20.0,15.0,23.0,21.0,21.142857,23.400000,5.984106,0
493016,2017-12-28,10,27.0,27.0,20.0,25.0,28.0,21.428571,23.366667,6.241184,0
493017,2017-12-29,10,27.0,16.0,27.0,22.0,31.0,20.571429,22.866667,6.553807,0
493018,2017-12-30,10,27.0,28.0,16.0,29.0,35.0,20.428571,22.633333,6.347103,0


In [11]:
Y_train = train['sales']
X_train = train.drop('sales', axis = 1)

In [12]:
import holidays
years = X_train['date'].dt.year.unique()
years
holi = []
for h in holidays.UnitedStates(years=years).items():
  holi.append(h[0])
holi
holiday_set = set(holi)
holiday_set

{datetime.date(2013, 1, 1),
 datetime.date(2013, 1, 21),
 datetime.date(2013, 2, 18),
 datetime.date(2013, 5, 27),
 datetime.date(2013, 7, 4),
 datetime.date(2013, 9, 2),
 datetime.date(2013, 10, 14),
 datetime.date(2013, 11, 11),
 datetime.date(2013, 11, 28),
 datetime.date(2013, 12, 25),
 datetime.date(2014, 1, 1),
 datetime.date(2014, 1, 20),
 datetime.date(2014, 2, 17),
 datetime.date(2014, 5, 26),
 datetime.date(2014, 7, 4),
 datetime.date(2014, 9, 1),
 datetime.date(2014, 10, 13),
 datetime.date(2014, 11, 11),
 datetime.date(2014, 11, 27),
 datetime.date(2014, 12, 25),
 datetime.date(2015, 1, 1),
 datetime.date(2015, 1, 19),
 datetime.date(2015, 2, 16),
 datetime.date(2015, 5, 25),
 datetime.date(2015, 7, 3),
 datetime.date(2015, 7, 4),
 datetime.date(2015, 9, 7),
 datetime.date(2015, 10, 12),
 datetime.date(2015, 11, 11),
 datetime.date(2015, 11, 26),
 datetime.date(2015, 12, 25),
 datetime.date(2016, 1, 1),
 datetime.date(2016, 1, 18),
 datetime.date(2016, 2, 15),
 datetime.dat

In [13]:
X_train['month'] = X_train['date'].dt.month
X_train['daysofweek'] = X_train['date'].dt.dayofweek
X_train['isWeekend'] = X_train['date'].dt.dayofweek > 4
X_train['isHoliday'] = X_train['date'].isin(holiday_set)
X_train

/tmp/ipython-input-617757616.py:4: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  X_train['isHoliday'] = X_train['date'].isin(holiday_set)


,date,store,item,lag_sales_1,lag_sales_7,lag_sales_30,rolling_mean_7,rolling_mean_30,rolling_std_7,is_promotion,month,daysofweek,isWeekend,isHoliday
30,2013-01-31,1,1.0,9.0,8.0,13.0,11.000000,10.500000,2.708013,0,1,3,False,False
31,2013-02-01,1,1.0,13.0,14.0,11.0,10.571429,10.500000,2.370453,0,2,4,False,False
32,2013-02-02,1,1.0,11.0,12.0,14.0,11.857143,10.733333,4.634241,1,2,5,True,False
33,2013-02-03,1,1.0,21.0,12.0,13.0,12.285714,10.800000,4.785892,0,2,6,True,False
34,2013-02-04,1,1.0,15.0,11.0,10.0,12.714286,10.933333,4.785892,0,2,0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493015,2017-12-27,10,27.0,15.0,23.0,21.0,21.142857,23.400000,5.984106,0,12,2,False,False
493016,2017-12-28,10,27.0,20.0,25.0,28.0,21.428571,23.366667,6.241184,0,12,3,False,False
493017,2017-12-29,10,27.0,27.0,22.0,31.0,20.571429,22.866667,6.553807,0,12,4,False,False
493018,2017-12-30,10,27.0,16.0,29.0,35.0,20.428571,22.633333,6.347103,0,12,5,True,False


In [14]:
X_train.isnull().sum()

,0
date,0
store,0
item,0
lag_sales_1,0
lag_sales_7,0
lag_sales_30,0
rolling_mean_7,0
rolling_mean_30,0
rolling_std_7,0
is_promotion,0


Lets try Gradient Descent.

In [15]:
from sklearn.linear_model import SGDRegressor

In [16]:
new_X_train = X_train.drop('date', axis='columns')

In [17]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(new_X_train,Y_train,test_size=0.2,random_state=2)

In [18]:
reg = SGDRegressor(max_iter=100,learning_rate='invscaling',eta0=0.0001)

In [19]:
reg.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_stochastic_gradient.py:1608: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


SGDRegressor(eta0=0.0001, max_iter=100)

In [20]:
reg.predict(X_test)

array([75.03895665, 62.94866909,  7.4529853 , ..., 66.0238062 ,
       13.24224014, 99.82824295])

In [21]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [22]:
r2_score(y_test, reg.predict(X_test))

0.9389088213459373

In [23]:
reg.coef_

array([-5.14846463e-03, -4.23920628e-03, -8.55754382e-02,  1.53101525e-01,
        9.92476166e-03,  1.09513231e+00, -1.75551783e-01, -5.65134379e-02,
        1.10312277e+01, -7.22630054e-02,  2.46409098e+00,  4.65361418e-01,
       -7.01120865e-02])

Lets scale the features and see the results of R2 Score

In [24]:
import numpy as np

In [25]:
# Convert to dummies for 'store', 'item', 'isWeekend', and 'isHoliday'
# Ensure consistency with how test features will be created.
X_train_final = pd.get_dummies(X_train, columns=['store', 'item', 'isWeekend', 'isHoliday'], drop_first=True)
X_test_final = pd.get_dummies(X_test, columns=['store', 'item', 'isWeekend', 'isHoliday'], drop_first=True)

In [26]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

In [27]:
reg1 = SGDRegressor(max_iter=100,learning_rate='invscaling',eta0=0.0001)
reg1.fit(X_train_scaled, y_train)
reg1.predict(X_test_scaled)
r2_score(y_test, reg1.predict(X_test_scaled))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_stochastic_gradient.py:1608: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


0.937310623771919

Stochastic Gradient Descent performed a bit well after scaling the features. Lets try Mini Batch Gradient Descent

In [28]:
sgd = SGDRegressor(learning_rate='invscaling',eta0=0.001)

In [29]:
import random
batch_size = 256

for i in range(50):

    idx = random.sample(range(X_train_scaled.shape[0]),batch_size)
    sgd.partial_fit(X_train_scaled[idx],y_train.iloc[idx])

In [30]:
sgd.coef_

array([ 4.14812232,  5.51825557,  3.3727016 ,  5.49888644,  4.95276832,
        3.07294396,  2.27423565, -0.8039236 ,  3.17918481,  0.89673887,
        0.35274945,  0.09541154, -0.24583684, -0.3965774 , -0.54433811,
        0.26268663,  0.1397428 ,  0.04174067,  0.3549147 , -0.20486981,
       -1.04377621, -0.60397246,  0.12185176,  0.06458943,  0.64246525,
        0.33004636,  0.37304109,  0.50272091,  0.37619988,  0.78112793,
        0.28818538,  1.14819399, -0.58381252, -0.15832595,  0.8033451 ,
       -0.37640058, -0.35156076, -0.65105196,  0.77309804, -0.58224719,
        0.34817542,  0.88853878, -0.12699408, -1.00443018,  0.78186005,
        1.14910446, -0.64006282])

In [31]:
sgd.intercept_

array([43.5117873])

In [32]:
sgd.predict(X_test_scaled)

array([62.68256479, 57.18825252, -6.45939473, ..., 62.19438973,
        0.98345617, 91.94261369])

In [33]:
r2_score(y_test, sgd.predict(X_test_scaled))

0.7796481270797819

Now lets apply Mini Batch Gradient Descent from Scratch.

In [34]:
import numpy as np
import random

class MBGDRegressorScratch:
    def __init__(self, batch_size, learning_rate=0.01, max_iter=100):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.batch_size = batch_size
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train, y_train):
        # Convert to numpy arrays once to avoid .iloc overhead inside loops
        X = X_train.values
        y = y_train.values

        # 1. INITIALIZE WITH ZEROS (Crucial to prevent overflow)
        self.intercept_ = 0.0
        self.coef_ = np.zeros(X.shape[1])

        for i in range(self.max_iter):
            # Create a list of indices and shuffle for true SGD
            indices = np.arange(X.shape[0])
            np.random.shuffle(indices)

            for start in range(0, X.shape[0], self.batch_size):
                idx = indices[start:start + self.batch_size]
                X_batch = X[idx]
                y_batch = y[idx]

                y_hat = np.dot(X_batch, self.coef_) + self.intercept_
                error = y_batch - y_hat

                # 2. CALCULATE GRADIENTS
                intercept_der = -2 * np.mean(error)
                # Normalize coef_der by batch size to keep gradients stable
                coef_der = -2 * np.dot(error, X_batch) / self.batch_size

                # 3. UPDATE (Lower learning rate helps stability)
                self.intercept_ -= self.learning_rate * intercept_der
                self.coef_ -= self.learning_rate * coef_der

            # Check for NaN to stop early if it explodes
            if np.isnan(self.coef_).any():
                print(f"Exploded at iteration {i}. Lower your learning rate!")
                break
    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_




In [35]:
# Run with a smaller learning rate and scaled data
mbr = MBGDRegressorScratch(batch_size=256, learning_rate=0.001, max_iter=50)
mbr.fit(pd.DataFrame(X_train_scaled), y_train)

In [36]:
mbr.predict(X_test_scaled)

array([74.97025517, 64.01020298,  7.74862338, ..., 66.65979775,
       12.97445433, 99.71507079])

In [37]:
r2_score(y_test, mbr.predict(X_test_scaled))

0.9388003812214234

In [38]:
mbr.coef_

array([-2.15745432e+00,  4.36151718e+00, -3.27566354e-01,  2.62968141e+01,
        3.08889692e-01, -1.68191859e-01,  2.71380101e+00, -4.67665106e-01,
        4.95946243e+00, -2.39962733e-02,  1.42150666e-02,  5.30867065e-03,
        1.03703470e-02,  7.77588643e-03,  2.16029566e-03, -2.36102233e-02,
       -7.80414428e-03, -1.10149034e-02, -1.21000160e-02, -2.12685776e-02,
       -1.83125740e-02, -7.88263216e-03, -1.41380035e-02, -2.26013560e-02,
       -2.39497893e-02, -1.07885886e-02, -2.28816119e-02, -1.83772793e-02,
       -1.37395172e-02,  1.99743794e-03,  5.17337171e-04, -3.14814554e-02,
       -2.49657319e-03, -2.52350644e-02, -2.35318641e-02,  3.84630875e-03,
       -2.99870732e-02,  1.06601546e-02, -1.48178199e-02, -1.84448328e-02,
       -1.06126016e-02, -3.00290980e-02, -7.67351790e-03, -2.14143882e-02,
       -3.58975305e-03,  1.29034431e-01, -2.58735253e-03])